In [1]:
import random
import pickle
import numpy as np
from collections import defaultdict
import scipy.sparse as sp

import os
import random
import pandas as pd
import json
import pickle
import gzip
from tqdm import tqdm

def load_pickle(filename):
    with open(filename, "rb") as f:
        return pickle.load(f)


def save_pickle(data, filename):
    with open(filename, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

def load_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)
    
def ReadLineFromFile(path):
    lines = []
    with open(path,'r') as fd:
        for line in fd:
            lines.append(line.rstrip('\n'))
    return lines

def save_pickle(data, filename):
    with open(filename, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

def parse(path):
    g = gzip.open(path, 'r')
    for l in g:
        # Convert bytes to string
        data_str = l.decode('utf-8')
        # Replace 'false' with 'False' and 'true' with 'True'
        data_str = data_str.replace('false', 'False').replace('true', 'True')

        # Parse the JSON string into a dictionary
        yield eval(data_str)
'''
Set seeds
'''
seed = 999
random.seed(seed)
np.random.seed(seed)

In [2]:
DATA_PATH = '../data/'

In [3]:
DATASET = 'clothing'

In [4]:
test_samples = ReadLineFromFile(os.path.join(DATA_PATH, DATASET, 'negative_samples.txt'))
len(test_samples)

39387

In [5]:
sequential_data = ReadLineFromFile(os.path.join(DATA_PATH, DATASET, 'sequential_data.txt'))
item_count = defaultdict(int)
user_items = defaultdict()

for line in sequential_data:
    user, items = line.strip().split(' ', 1)
    items = items.split(' ')
    items = [str(item) for item in items]
    user_items[user] = items
    for item in items:
        item_count[item] += 1

In [6]:
user_items['1']

['1', '2', '3', '4', '5']

In [7]:
all_item = list(item_count.keys())

In [8]:
all_item[:4]

['1', '2', '3', '4']

In [9]:
datamaps = load_json(os.path.join(DATA_PATH, DATASET, 'datamaps.json'))
user2id = datamaps['user2id']
item2id = datamaps['item2id']
user_list = list(datamaps['user2id'].keys())
item_list = list(datamaps['item2id'].keys())
id2item = datamaps['id2item']
id2user = datamaps['id2user']

In [10]:
list(id2user.keys())[:4]

['1', '2', '3', '4']

In [11]:
print("#samples:",len(test_samples[0].split(' ',1)[1].split(' ')))
test_samples[0].split(' ',1)[1].split(' ')[:10]

#samples: 99


['16117',
 '4592',
 '20175',
 '20480',
 '7944',
 '13878',
 '19108',
 '840',
 '16686',
 '10884']

In [12]:
train_negative = []
for user in tqdm(list(id2user.keys())):
    user_seq = user_items[user][:]
    user_seq = set([str(x) for x in user_seq])
    candidate_samples = []
    candidate_num = len(user_seq)
    already_samples = test_samples[int(user)-1].split(' ', 1)[1].split(' ')
    already_samples = set([str(x) for x in already_samples])
    while len(candidate_samples) < candidate_num:
        choices = [item for item in all_item if item not in user_seq]
        sample_ids = np.random.choice(choices, candidate_num, replace=False)
        sample_ids = [str(item) for item in sample_ids if (item not in user_seq)]
        sample_ids = [str(item) for item in sample_ids if (item not in already_samples)]
        sample_ids = [str(item) for item in sample_ids if (item not in candidate_samples)]
        candidate_samples.extend(sample_ids)
    candidate_samples = candidate_samples[:candidate_num]
    train_negative.append([user] + candidate_samples)
    

100%|██████████| 39387/39387 [01:26<00:00, 457.11it/s]


In [13]:
train_negative[0], user_items['1'], test_samples[0]

(['1', '22988', '1897', '12841', '18329', '19638'],
 ['1', '2', '3', '4', '5'],
 '1 16117 4592 20175 20480 7944 13878 19108 840 16686 10884 22332 7328 7343 7471 22651 15429 13754 14101 1996 16720 2277 17638 22771 7891 19905 12113 2463 8915 9331 11949 20220 15359 1136 1879 13352 4473 1839 13392 95 4486 23016 17223 1114 746 3936 12751 2364 2677 1811 12109 1453 17038 10891 16491 19423 14028 13594 1981 782 13268 10206 22814 16967 8037 9371 621 5654 18844 17729 19662 13753 12490 2190 2178 7592 89 15620 11028 15619 6310 11908 12408 11657 17735 6305 19738 22731 16739 22078 18088 3489 9676 3247 10509 1845 13513 10649 6453 21301')

In [14]:
len(train_negative)

39387

# Check overlaps

In [15]:
test_negs = []
for idx in range(len(test_samples)):
    lst = test_samples[idx].split(' ')[1:]
    user = test_samples[idx].split(' ')[0]
    for val in lst:
        test_negs.append((int(user) - 1, int(val)))
len(test_negs)

3899313

In [16]:
test_negs[:10]

[(0, 16117),
 (0, 4592),
 (0, 20175),
 (0, 20480),
 (0, 7944),
 (0, 13878),
 (0, 19108),
 (0, 840),
 (0, 16686),
 (0, 10884)]

In [17]:
train_negs = []
for idx in range(len(train_negative)):
    lst = train_negative[idx]
    user = lst[0]
    items = lst[1:]
    for item in items:
        train_negs.append((int(user)-1,int(item)))

In [18]:
len(train_negs)

278677

In [19]:
train_negs[:10]

[(0, 22988),
 (0, 1897),
 (0, 12841),
 (0, 18329),
 (0, 19638),
 (1, 11299),
 (1, 10199),
 (1, 18093),
 (1, 17669),
 (1, 6206)]

In [20]:
inter = []
for user,items in user_items.items():
    new_user = int(user) - 1
    for item in items:
        item = int(item)
        inter.append((new_user,item))

In [21]:
len(inter)

278677

In [22]:
inter[:10]

[(0, 1),
 (0, 2),
 (0, 3),
 (0, 4),
 (0, 5),
 (1, 6),
 (1, 7),
 (1, 8),
 (1, 9),
 (1, 10)]

In [23]:
len(set(train_negs))

278677

In [24]:
len(set(test_negs))

3899309

In [25]:
len(set(inter))

278677

In [26]:
common1 = set(train_negs).intersection(set(test_negs))
len(common1)

0

In [27]:
common2 = set(train_negs).intersection(set(inter))
len(common2)

0

In [28]:
common2

set()

In [29]:
save_pickle(train_negative,os.path.join(DATA_PATH,DATASET,'train-negatives.pkl'))